In [ ]:
pip install -U git+https://github.com/facebookresearch/demucs

In [2]:
# pip uninstall torch torchaudio torchvision -y

In [3]:
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [48]:
import torch
import numpy as np
import sounddevice as sd
import gc
import pyaudio
import wave
import subprocess as sp
import sys
import threading
import tempfile
from pathlib import Path
import soundfile as sf
import io
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from datetime import datetime
from docx import Document
import json
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import re
import os
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
import logging
import shutil

In [32]:
logger = logging.getLogger(__name__)

In [5]:
# Check GPU setup
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [6]:
gc.collect()  # Clean up CPU memory
torch.cuda.empty_cache()  # Clean up GPU memory

In [7]:
# Load the entire model
model = torch.load("whisper_model.pth",weights_only=False)

In [8]:
# Move model to GPU if available
if torch.cuda.is_available():
    model = model.to("cuda")
else:
    print("Error Loading Model to GPU")

In [9]:
# Verifying the model in gpu
print(next(model.parameters()).device)

cuda:0


In [10]:
# Audio configuration
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024

In [11]:
# Demucs configuration
MODEL = "htdemucs"
TWO_STEMS = 'vocals'

In [ ]:
def process_with_demucs(temp_file: Path):
    with tempfile.TemporaryDirectory() as tmp_output_dir:
        output_dir = Path(tmp_output_dir)
        cmd = [
            "python", "-m", "demucs.separate",
            "-n", MODEL,
            "--two-stems", TWO_STEMS,
            "-o", str(output_dir),
            str(temp_file)
        ]

        process = sp.Popen(cmd, stdout=sp.PIPE, stderr=sp.PIPE)

        def stream_reader(stream, prefix):
            while True:
                line = stream.readline()
                if not line:
                    break
                print(f"[{prefix}] {line.decode().strip()}")

        stdout_thread = threading.Thread(target=stream_reader, args=(process.stdout, "DEMUCS"))
        stderr_thread = threading.Thread(target=stream_reader, args=(process.stderr, "ERROR"))

        stdout_thread.start()
        stderr_thread.start()
        process.wait()

        if process.returncode != 0:
            print(f"\n Separation failed (code {process.returncode})")
            return None

        print("\n Vocal separation successful!")

        # Find the folder where Demucs saved the outputs
        track_name = temp_file.stem
        model_folder = output_dir / MODEL / track_name

        vocals_path = model_folder / "vocals.wav"
        no_vocals_path = model_folder / "no_vocals.wav"

        # Load files into memory
        vocals_bytes = io.BytesIO(vocals_path.read_bytes())
        no_vocals_bytes = io.BytesIO(no_vocals_path.read_bytes())

        return {
            "vocals": vocals_bytes,
        }

In [ ]:
def record_microphone() -> Path:
    audio = pyaudio.PyAudio()
    stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)

    frames = []
    stop_recording = False

    print("\n Recording... (Press Q to stop recording)")

    def listen_for_quit():
        nonlocal stop_recording
        while True:
            if input().strip().lower() == 'q':
                stop_recording = True
                print("\n Stopping recording...")
                break

    threading.Thread(target=listen_for_quit, daemon=True).start()

    try:
        while not stop_recording:
            data = stream.read(CHUNK, exception_on_overflow=False)
            audio_data = np.frombuffer(data, dtype=np.int16)
            
            # Real-time processing
            peak = np.max(np.abs(audio_data))
            if peak > 0:
                audio_data = (audio_data / peak * 32767).astype(np.int16)

            frames.append(data)
    except KeyboardInterrupt:
        print("\n Stopping recording...")
    finally:
        stream.stop_stream()
        stream.close()
        audio.terminate()
    
    # Create in-memory WAV file
    # Save to temp WAV file
    temp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    with wave.open(temp_wav, 'wb') as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(audio.get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(b''.join(frames))

        # Process directly from memory
        result = process_with_demucs(Path(temp_wav.name))
        return result  

In [ ]:
def Translation():
    while True:
        choice = input("\nPress ENTER to start recording (Q to quit): ").strip().lower()
        if choice == 'q':
            print("\n Exiting...")
            break

        try:
            vocal = record_microphone()
            if vocal is None:
                print("Recording or separation failed.")
                continue

            vocals_wav = vocal["vocals"]
            vocals_wav.seek(0)

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_audio_file:
                temp_audio_file.write(vocals_wav.read())
                temp_audio_path = temp_audio_file.name

            result = model.transcribe(temp_audio_path)
            print("\n Transcription:", result["text"])

        except Exception as e:
            print(f"\n Error occurred: {str(e)}")

        # Ask again after the recording has finished or stopped
        continue_choice = input("\nPress ENTER to record again (Q to quit): ").strip().lower()
        if continue_choice == 'q':
            print("\n Exiting...")
            break
    
    return result['text']
        

In [17]:
llm = OllamaLLM(
    model="deepseek-r1:latest"
)

In [18]:
current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [19]:
prompt_template = PromptTemplate(
    input_variables=["text", "current_datetime"],
    template="""
<|user|>
Your task is to format the given transcription text with the following **strict guidelines**:

1. Correct ONLY grammar, punctuation, and sentence structure errors in the provided text.
2. Do NOT paraphrase, reorder sentences, or change any wording unless it's grammatically incorrect.
3. Generate **one appropriate main heading** that reflects the overall context of the content.
4. Provide a concise **Overview section** summarizing the full context in 2-3 sentences.
5. Present the corrected full content after the Overview section, ensuring NO changes other than grammar/punctuation fixes.
6. Output format MUST be as follows (EXACT STRUCTURE REQUIRED):
   - Start with **Main Heading:** followed by your heading (e.g., **Main Heading:** Elon Musk: A Visionary Entrepreneur) no characters or symbols before **
   - Next line: **Overview:** followed by your 2-3 sentence summary no characters or symbols before **
   - Next line: **Corrected Full Content:** followed by the original text with grammar/punctuation fixes no characters or symbols before **
   - Final line: **Log Date and Time:** {current_datetime} no characters or symbols before **

⚠️ Critical Instructions:
- Use the EXACT labels **Main Heading:**, **Overview:**, **Corrected Full Content:**, and **Log Date and Time:**.
- Do NOT use variations like "Content" or "Log" instead of the required labels.
- Do NOT add extra text, explanations, or formatting (e.g., ---, **bold**, or bullet points).
- Keep the corrected content in paragraph form.
- If no corrections are needed, output the original text under **Corrected Full Content:**.

Log: {current_datetime}
---

Now, apply the formatting to the following transcription:

Text:
{text}
<|assistant|>
"""
)

In [20]:
chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
)

C:\Users\azefr\AppData\Local\Temp\ipykernel_2580\2628146452.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


In [21]:
transcribed_text = Translation()


🎤 Recording... (Press Q to stop recording)

⏹️ Stopping recording...
[DEMUCS] Selected model is a bag of 1 models. You will see that many progress bars per track.
[DEMUCS] Separated tracks will be stored in C:\Users\azefr\AppData\Local\Temp\tmpatwlhmsq\htdemucs
[DEMUCS] Separating track C:\Users\azefr\AppData\Local\Temp\tmp3rjawux6.wav
100%|████████████████████████████████████████████████| 81.89999999999999/81.89999999999999 [00:08<00:00,  9.77seconds/s]nds/s]

✅ Vocal separation successful!

📝 Transcription:  Elon Musk is a billionaire, entrepreneur, inventor and engineer best known for founding and leading several innovation companies including Tesla, SpaceX, Neuralink and the Boring Company. He has been a driving force behind the development of electric vehicles, renewable energy solutions and space exploration. Musk's ambition to make life multi-planetary through SpaceX's goal of colonizing Mars has granted widespread attention. He is also deeply involved in the advancement of art

In [22]:
transcribed_text

" Elon Musk is a billionaire, entrepreneur, inventor and engineer best known for founding and leading several innovation companies including Tesla, SpaceX, Neuralink and the Boring Company. He has been a driving force behind the development of electric vehicles, renewable energy solutions and space exploration. Musk's ambition to make life multi-planetary through SpaceX's goal of colonizing Mars has granted widespread attention. He is also deeply involved in the advancement of artificial intelligence, brain machine, interfaces and tunnel transportation system, knowing for his bold vision and often controversial statements. Musk has become one of the most influential figures in the tech and innovation sectors, shaping industries and challenging conventional ways of thinking. Musk has become one of the most influential figures in the world. My name is Vimal VK and this is purely for testing purpose. This project is speech to text recognition using OpenAI Whisper model."

In [23]:
type(transcribed_text)

str

In [24]:
handler = StreamingStdOutCallbackHandler()
result = chain.invoke({"text": transcribed_text,"current_datetime":current_datetime}, callbacks=[handler])
print(result["text"])

<think>
Okay, I need to help format the user's transcription according to their guidelines. First, let me read through the instructions carefully.

The user wants only grammar, punctuation, and sentence structure errors fixed without paraphrasing or changing any wording unless it's wrong. They also want one main heading that reflects the content, an Overview section summarizing in 2-3 sentences, and then the corrected text with specific formatting labels.

Looking at the provided text:

1. The user mentions Elon Musk is a billionaire, entrepreneur, etc., but there's no period after 'billionaire'.
2. "best known for founding and leading several innovation companies" should be separated by a comma.
3. Missing punctuation in "He has been..." and "is deeply involved..."
4. Extra words like "My name is Vimal VK and this is purely for testing purpose." seem unnecessary.

So, I'll fix those issues. The main heading could be about his influence in tech innovation. Then, the overview will be co

In [25]:
# Extract the raw text from the result
raw_text = result['text']

# Initialize a dictionary to hold the parsed sections
parsed_data = {
    "Main Heading": "",
    "Overview": "",
    "Corrected Content": "",
    "Log Date and Time": ""
}

# Split the text into lines and process
current_section = None
lines = raw_text.split('\n')
for line in lines:
    line = line.strip()
    
    # Detect main heading
    if line.startswith('**Main Heading:**'):
        parsed_data["Main Heading"] = line.replace('**Main Heading:**', '').strip()
        current_section = None
        
    # Detect Overview section
    elif line.startswith('**Overview:**'):
        current_section = 'Overview'
        parsed_data[current_section] = line.replace('**Overview:**', '').strip()
        
    # Detect Corrected Content section
    elif line.startswith('**Corrected Full Content:**'):
        current_section = 'Corrected Content'
        parsed_data[current_section] = line.replace('**Corrected Full Content:**', '').strip()
        
    # Detect Log Date and Time
    elif line.startswith('**Log Date and Time:**'):
        parsed_data["Log Date and Time"] = line.replace('**Log Date and Time:**', '').strip()
        current_section = None
        
    # Handle multi-line sections
    elif current_section:
        parsed_data[current_section] += line.strip()

# Convert to JSON
json_output = json.dumps(parsed_data, indent=4,ensure_ascii=False)
print(json_output)

{
    "Main Heading": "Elon Musk: A Visionary Figure in Tech Innovation",
    "Overview": "Elon Musk is a renowned billionaire entrepreneur known for founding several innovative companies like Tesla and SpaceX. He has ambitious plans to expand space exploration and integrate artificial intelligence into daily life, significantly impacting the tech industry.",
    "Corrected Content": "Elon Musk is a billionaire, entrepreneur, inventor, and engineer best known for founding and leading several innovation companies including Tesla, SpaceX, NeuralLink, and the Boring Company. He has been a driving force behind the development of electric vehicles, renewable energy solutions, and space exploration. Musk's ambition to make life multi-planetary through SpaceX's goal of colonizing Mars has garnered widespread attention. He is deeply involved in advancing artificial intelligence, brain-machine interfaces, and tunnel transportation systems, known for his bold vision and controversial statements.

In [26]:
json_output = json.loads(json_output)

In [30]:
# Create a new Word document
doc = Document()

# Add Main Heading (centered and bold)
main_heading = doc.add_paragraph()
main_heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = main_heading.add_run(json_output["Main Heading"])
run.bold = True
run.font.size = Pt(14)

# Add Overview section
doc.add_paragraph()  # Spacer
overview_heading = doc.add_paragraph("Overview")
overview_heading.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
overview_heading.runs[0].bold = True
doc.add_paragraph(json_output["Overview"]).alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

# Add Corrected Content section
doc.add_paragraph()  # Spacer
content_heading = doc.add_paragraph("Content")
content_heading.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
content_heading.runs[0].bold = True
doc.add_paragraph(json_output["Corrected Content"]).alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

# Add Log Date and Time (bottom-right)
doc.add_paragraph()  # Spacer
log_paragraph = doc.add_paragraph()
log_paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
log_run = log_paragraph.add_run(f"Log Date and Time: {json_output['Log Date and Time']}")
log_run.bold = True

# Save the document
# Remove invalid filename characters from Main Heading
safe_heading = re.sub(r'[<>:"/\\|?*]', '_', json_output["Main Heading"])

# Clean up the datetime to be filename-safe
safe_datetime = current_datetime.replace(":", "-").replace(" ", "_")

# Create the safe filenames
safe_docx_filename = f"{safe_heading}__{safe_datetime}.docx"
safe_pdf_filename = f"{safe_heading}__{safe_datetime}.pdf"

# Save the Word document temporarily
doc.save(safe_docx_filename)

# Ensure the absolute file path is used
doc_path = os.path.abspath(safe_docx_filename)
pdf_path = os.path.abspath(safe_pdf_filename)

# Convert the Word document to PDF
try:
    import comtypes.client

    # Initialize COM for Word application
    word = comtypes.client.CreateObject("Word.Application")
    word.Visible = False

    # Open the saved Word document using the absolute path
    doc_obj = word.Documents.Open(doc_path)

    # Save as PDF
    doc_obj.SaveAs(pdf_path, FileFormat=17)  # 17 corresponds to PDF format
    doc_obj.Close()
    word.Quit()

    # Delete the temporary .docx file
    if os.path.exists(doc_path):
        os.remove(doc_path)

    print(f"PDF saved successfully as '{safe_pdf_filename}'.")
except Exception as e:
    print(f"Error converting to PDF: {e}")

PDF saved successfully as 'Elon Musk_ A Visionary Figure in Tech Innovation__2025-05-02_07-56-21.pdf'.


In [29]:
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\azefr\AppData\Local\Temp\ipykernel_2580\3738465760.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\azefr\OneDrive\Desktop\Research Assistant\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
def pdf_reader(file_path):
    """Reads and splits a PDF document into smaller chunks for processing."""
    try:
        logger.info(f"Loading PDF: {file_path}")
        loader = PyPDFLoader(file_path)
        documents = loader.load()

        # Split into chunks for better retrieval
        text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        docs = text_splitter.split_documents(documents)

        logger.info(f"PDF processing complete: {len(docs)} chunks created")
        return docs
    except Exception as e:
        logger.error(f"Error reading PDF: {e}")
        return []


In [36]:
pdf_files = [os.path.abspath(file) for file in os.listdir() if file.endswith('.pdf')]
pdf_files

['c:\\Users\\azefr\\OneDrive\\Desktop\\Research Assistant\\Elon Musk_ A Visionary Figure in Tech Innovation__2025-05-02_07-56-21 copy 2.pdf',
 'c:\\Users\\azefr\\OneDrive\\Desktop\\Research Assistant\\Elon Musk_ A Visionary Figure in Tech Innovation__2025-05-02_07-56-21 copy.pdf',
 'c:\\Users\\azefr\\OneDrive\\Desktop\\Research Assistant\\Elon Musk_ A Visionary Figure in Tech Innovation__2025-05-02_07-56-21.pdf']

In [50]:
def rag_deep_seek(query,pdf_name=None):
    curr_dir = 'c:\\Users\\azefr\\OneDrive\\Desktop\\Research Assistant\\'
    try:
        if pdf_name:
            # Process PDF and use RAG
            path = os.path.join(curr_dir, pdf_name)
            docs = pdf_reader(path)
            logger.info("PDF processed successfully")
            
            # Create vector store
            vector_store = FAISS.from_documents(docs, embeddings)
            logger.info("Vector store created successfully")

            # Create RAG chain
            qa_chain = RetrievalQA.from_chain_type(
                llm=llm,
                retriever=vector_store.as_retriever(),
                chain_type="stuff",
            )

            # Modify the query with instructions
            modified_query = query + """
                You are an expert chatbot answering questions based on retrieved documents and also general knowledge.\n
                If the answer is not in the documents, respond with 'I don't have enough information to answer that question.'\n
            """
            response = qa_chain.invoke(modified_query)
            response_result = response['result']

        else:

            prompt_template = """You are a helpful AI assistant. Answer the following question based on your knowledge:
            Question: {query}
            Answer:"""
            prompt = PromptTemplate.from_template(prompt_template)
            llm_chain = LLMChain(llm=llm, prompt=prompt)
            response = llm_chain.run(query)
            response_result = response.strip()
        
        logger.info(f"Response generated: {response_result}")
        return response_result
    
    except Exception as e:
        logger.error(f"Error in rag_deep_seek function: {e}")
        return "An error occurred while processing your request."

In [49]:
def input_querycall():
    curr_dir = 'c:\\Users\\azefr\\OneDrive\\Desktop\\Research Assistant\\'
    print('1. General.')
    print('2. Using Document Name')
    print('3. Chat Using External Data.')
    inp = input('Enter your choice: ')
    response = None
    
    if inp == '1':
        query = input("Ask: ")
        if query:
            response = rag_deep_seek(query)
    elif inp == '2':
        query = input("Ask: ")
        pdf_name = input("Enter PDF Name: ")
        if query and pdf_name:
            response = rag_deep_seek(query, pdf_name)
    elif inp == '3':
        pdf_path = input("Enter the path to the PDF file to upload: ")
        if os.path.isfile(pdf_path):
            pdf_name = os.path.basename(pdf_path)
            dest_path = os.path.join(curr_dir, pdf_name)
            try:
                shutil.copy(pdf_path, dest_path)
                print(f"PDF '{pdf_name}' uploaded successfully to {dest_path}.")
                query = input("Ask your question: ")
                if query:
                    response = rag_deep_seek(query, pdf_name)
            except Exception as e:
                print(f"Error uploading PDF: {e}")
                response = None
        else:
            print("Error: The specified file does not exist.")
            response = None
    else:
        print("Invalid choice.")
    
    return response

In [51]:
def clean_think_tags(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

In [46]:
res = input_querycall()
print(res)

<think>
Alright, so I need to help the user by summarizing a document about Elon Musk. Let me read through the context carefully.

First, the document starts by introducing Elon Musk as a renowned billionaire entrepreneur known for companies like Tesla and SpaceX. It mentions his ambitious plans in space exploration and AI integration, which are big impacts on the tech industry.

Looking deeper into the content section, it provides more details about the companies he founded: Tesla, SpaceX, NeuralLink, and the Boring Company. These are all significant ventures that push innovation in electric vehicles, renewable energy, space exploration, AI, brain-machine interfaces, and tunnel transportation.

The log date is May 2, 2025 at 7:56:21 AM, but I don't think that adds much to the summary beyond indicating a recent document. 

I need to make sure my summary includes his key accomplishments—like founding major companies—and his future goals in space and AI. Also, it's important to note how 

In [52]:
cleaned_res = clean_think_tags(res)
print(cleaned_res)

Elon Musk is a renowned billionaire entrepreneur known for founding several innovative companies such as Tesla, SpaceX, NeuralLink, and the Boring Company. He has ambitious plans to expand space exploration through SpaceX's goal of colonizing Mars and integrate artificial intelligence into daily life via Tesla and NeuralLink. His ventures significantly impact the tech industry, shaping it with bold visions and influencing various technological advancements.
